# Direct Lyapunov MPC Six-Scenario Study

This notebook is the canonical experiment entrypoint for the direct frozen-output-disturbance Lyapunov MPC path. It runs the six scenario matrix by default:

- `unbounded_hard`
- `bounded_hard`
- `unbounded_soft`
- `bounded_soft`
- `bounded_hard_u_prev`
- `bounded_soft_u_prev`

Each scenario uses the same plant, setpoint schedule, plant run mode, horizons, weights, and failure policy. The regularized bounded cases additionally turn on the `u_s-u_prev` target term. Hard infeasibility is recorded as a diagnostic result, not treated as a notebook failure.


## Setup

The visible switches below control the full study. Runtime artifacts are saved under `Data/debug_exports/direct_lyapunov_mpc_six_scenario/<timestamp>/<case_name>/`, and the final comparison artifacts are saved at the timestamped study root.


In [ ]:
import os
import shutil
from datetime import datetime
from pprint import pprint

import numpy as np
import pandas as pd
from IPython.display import Image, display

from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import compute_observer_gain
from Simulation.system_functions import PolymerCSTR
from Lyapunov.direct_lyapunov_mpc import (
    build_direct_lyapunov_run_bundle,
    design_direct_lyapunov_mpc_solver,
    direct_output_rmse_post_step,
    make_direct_lyapunov_comparison_record,
    run_direct_output_disturbance_lyapunov_mpc,
    save_direct_lyapunov_comparison_artifacts,
    save_direct_lyapunov_debug_artifacts,
)
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data


In [ ]:
predict_h = 9
cont_h = 3
rho_lyap = 0.98
lyap_eps = 1e-9
slack_penalty = 1e6
terminal_cost_scale = 0.0

n_tests = 2
set_points_len = 400
TEST_CYCLE = [False] * max(int(n_tests), 1)
warm_start = 0

save_case_bundles = True
save_case_plots = True
save_paper_plots = True
save_comparison_plots = True
copy_report_figures = True

# Start with the nominal plant by default. Change to "disturb" when the
# six cases are behaving correctly under nominal conditions.
plant_mode = "nominal"
disturbance_after_step = False

use_target_output_for_tracking = False
use_target_on_solver_fail = False
objective_steady_input_cost = False
objective_terminal_cost = False
target_u_prev_regularization_weight = 0.1


scenario_matrix = [
    {"case_name": "unbounded_hard", "target_mode": "unbounded", "lyapunov_mode": "hard"},
    {"case_name": "bounded_hard", "target_mode": "bounded", "lyapunov_mode": "hard"},
    {"case_name": "unbounded_soft", "target_mode": "unbounded", "lyapunov_mode": "soft"},
    {"case_name": "bounded_soft", "target_mode": "bounded", "lyapunov_mode": "soft"},
    {
        "case_name": "bounded_hard_u_prev",
        "target_mode": "bounded",
        "lyapunov_mode": "hard",
        "target_config": {"u_ref_weight": target_u_prev_regularization_weight},
    },
    {
        "case_name": "bounded_soft_u_prev",
        "target_mode": "bounded",
        "lyapunov_mode": "soft",
        "target_config": {"u_ref_weight": target_u_prev_regularization_weight},
    },
]

study_name = "direct_lyapunov_mpc_six_scenario"
study_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
study_root = os.path.join(os.getcwd(), "Data", "debug_exports", study_name, study_timestamp)

active_config = {
    "predict_h": predict_h,
    "cont_h": cont_h,
    "rho_lyap": rho_lyap,
    "lyap_eps": lyap_eps,
    "slack_penalty": slack_penalty,
    "terminal_cost_scale": terminal_cost_scale,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "scenario_matrix": scenario_matrix,
    "save_case_bundles": save_case_bundles,
    "save_case_plots": save_case_plots,
    "save_paper_plots": save_paper_plots,
    "save_comparison_plots": save_comparison_plots,
    "copy_report_figures": copy_report_figures,
    "plant_mode": plant_mode,
    "disturbance_after_step": disturbance_after_step,
    "use_target_output_for_tracking": use_target_output_for_tracking,
    "use_target_on_solver_fail": use_target_on_solver_fail,
    "objective_steady_input_cost": objective_steady_input_cost,
    "objective_terminal_cost": objective_terminal_cost,
    "target_u_prev_regularization_weight": target_u_prev_regularization_weight,
    "study_root": study_root,
}

print("Active direct Lyapunov MPC six-scenario configuration")
pprint(active_config)


## Plant And Controller Setup

This cell builds the polymer CSTR, output-disturbance augmented model, observer, input bounds, reward function, and direct Lyapunov MPC solver shared by all six scenarios.


In [ ]:
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=os.path.join("Data", "system_dict"),
    augmentation_style="rawlings",
    augmentation_mode="output_disturbance",
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

poles = np.array([
    0.44619852,
    0.33547649,
    0.36380595,
    0.70467118,
    0.3562966,
    0.42900673,
    0.4228262,
    0.96916776,
    0.91230187,
])
L = compute_observer_gain(A_aug, C_aug, poles)

u_ss_scaled = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
u_min_scaled = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
u_max_scaled = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])

u_dev_min = u_min_scaled - u_ss_scaled
u_dev_max = u_max_scaled - u_ss_scaled
bnds = tuple((float(lo), float(hi)) for lo, hi in zip(u_dev_min, u_dev_max)) * cont_h
IC_opt = np.zeros(inputs_number * cont_h)

Qy_diag = np.array([5.0, 1.0])
# Su_diag is used for Lyapunov terminal design only when objective_steady_input_cost=False.
Su_diag = np.array([1.0, 1.0])
Rdu_diag = np.array([1.0, 1.0])

reward_config, reward_fn = make_reward_fn_mpc_quadratic(Q_diag=Qy_diag, R_diag=Rdu_diag)

LMPC_obj = design_direct_lyapunov_mpc_solver(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag,
    NP=predict_h,
    NC=cont_h,
    Su_diag=Su_diag,
    u_min=u_dev_min,
    u_max=u_dev_max,
    Rdu_diag=Rdu_diag,
    terminal_set_on=True,
    terminal_alpha_scale=1.0,
    terminal_cost_scale=terminal_cost_scale,
    objective_steady_input_cost=objective_steady_input_cost,
    objective_terminal_cost=objective_terminal_cost,
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92


## Case Runner

`run_case(...)` is intentionally narrow: every scenario shares the same plant setup and controller settings. Only `target_mode`, `lyapunov_mode`, and optional target regularization vary.


In [ ]:
def run_case(case_spec):
    case_name = case_spec["case_name"]
    case_target_mode = case_spec["target_mode"]
    case_lyapunov_mode = case_spec["lyapunov_mode"]
    case_target_config = dict(case_spec.get("target_config", {}))
    case_config = {
        **active_config,
        "case_name": case_name,
        "target_mode": case_target_mode,
        "lyapunov_mode": case_lyapunov_mode,
        "target_config": case_target_config,
    }
    print(
        f"Running {case_name}: target_mode={case_target_mode}, "
        f"lyapunov_mode={case_lyapunov_mode}, target_config={case_target_config}"
    )

    cstr_case = PolymerCSTR(
        system_params,
        system_design_params,
        system_steady_state_inputs,
        delta_t,
        deviation_form=False,
    )
    results_case = run_direct_output_disturbance_lyapunov_mpc(
        system=cstr_case,
        LMPC_obj=LMPC_obj,
        y_sp_scenario=y_sp_scenario,
        n_tests=n_tests,
        set_points_len=set_points_len,
        steady_states=steady_states,
        IC_opt=IC_opt,
        bnds=bnds,
        L=L,
        data_min=data_min,
        data_max=data_max,
        test_cycle=TEST_CYCLE,
        reward_fn=reward_fn,
        nominal_qi=nominal_qi,
        nominal_qs=nominal_qs,
        nominal_ha=nominal_hA,
        qi_change=qi_change,
        qs_change=qs_change,
        ha_change=ha_change,
        target_mode=case_target_mode,
        lyapunov_mode=case_lyapunov_mode,
        target_config=case_target_config,
        target_H=None,
        mode=plant_mode,
        disturbance_after_step=disturbance_after_step,
        use_target_output_for_tracking=use_target_output_for_tracking,
        skip_terminal_if_alpha_small=True,
        alpha_terminal_min=1e-8,
        use_target_on_solver_fail=use_target_on_solver_fail,
        rho_lyap=rho_lyap,
        lyap_eps=lyap_eps,
        slack_penalty=slack_penalty,
        first_step_contraction_on=True,
        reset_system_on_entry=True,
        solver_options={"warm_start": True},
    )
    bundle_case = build_direct_lyapunov_run_bundle(
        source=case_name,
        results=results_case,
        steady_states=steady_states,
        config=case_config,
        data_min=data_min,
        data_max=data_max,
        extra={"reward_config": reward_config, "min_max_dict": min_max_dict},
    )

    debug_dir_case = None
    if save_case_bundles:
        debug_dir_case = save_direct_lyapunov_debug_artifacts(
            bundle_case,
            directory=study_root,
            prefix_name=case_name,
            save_plots=save_case_plots,
            save_paper_plots=save_paper_plots,
            timestamp_subdir=False,
        )
    record_case = make_direct_lyapunov_comparison_record(case_name, bundle_case, debug_dir_case)
    print(f"Completed {case_name}")
    pprint(record_case)
    return results_case, bundle_case, debug_dir_case, record_case


## Six-Scenario Study

This cell runs the full scenario matrix, saves per-case bundles and plots, then saves the comparison table and comparison plots at the study root.


In [ ]:
os.makedirs(study_root, exist_ok=True)

results_by_case = {}
bundles_by_case = {}
debug_dirs_by_case = {}
comparison_records = []

for case_spec in scenario_matrix:
    results_case, bundle_case, debug_dir_case, record_case = run_case(case_spec)
    case_name = case_spec["case_name"]
    results_by_case[case_name] = results_case
    bundles_by_case[case_name] = bundle_case
    debug_dirs_by_case[case_name] = debug_dir_case
    comparison_records.append(record_case)

comparison_paths = save_direct_lyapunov_comparison_artifacts(
    comparison_records,
    bundles_by_case,
    study_root,
    save_plots=save_comparison_plots,
)

comparison_df = pd.DataFrame(comparison_records)
comparison_df = comparison_df.sort_values(by="case_name").reset_index(drop=True)
display(comparison_df)
print("Study root:", study_root)
print("Comparison artifacts:")
pprint(comparison_paths)


## Report Figure Copy

When plotting is enabled, this cell copies comparison plots and key per-case plots into the tracked report figure folder. Runtime bundles remain under ignored `Data/debug_exports/`.


In [ ]:
report_figure_dir = os.path.join(os.getcwd(), "report", "figures", "direct_lyapunov_mpc_frozen_output_disturbance")
report_copied_files = []

if copy_report_figures and save_comparison_plots:
    os.makedirs(report_figure_dir, exist_ok=True)
    for _, src in comparison_paths.get("plot_paths", {}).items():
        if src and os.path.exists(src):
            dst = os.path.join(report_figure_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            report_copied_files.append(dst)

    key_case_plots = [
        "fig_mpc_outputs_full.png",
        "fig_mpc_inputs_full.png",
        "04_lyapunov_diagnostics.png",
        "05_target_diagnostics.png",
    ]
    for case_name, debug_dir in debug_dirs_by_case.items():
        plot_dir = None if debug_dir is None else os.path.join(debug_dir, "plots")
        if not plot_dir or not os.path.isdir(plot_dir):
            continue
        for name in key_case_plots:
            src = os.path.join(plot_dir, name)
            if os.path.exists(src):
                dst = os.path.join(report_figure_dir, f"{case_name}_{name}")
                shutil.copy2(src, dst)
                report_copied_files.append(dst)

print("Report figure directory:", report_figure_dir)
print("Copied report figures:")
for path in report_copied_files:
    print(path)


## Saved Artifact Preview

Display the comparison plots generated at the study root. Per-case plots and bundles are in each case folder printed above.


In [ ]:
display(comparison_df)

for name, path in comparison_paths.get("plot_paths", {}).items():
    if path and os.path.exists(path):
        print(name, path)
        display(Image(filename=path))
